In [1]:
!nvidia-smi

Wed Sep  2 17:55:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U transformers peft datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 16.3 MB/s eta 0:00:00


In [3]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model

In [4]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [6]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [7]:
data = {
    "question": [
        "What is Python?",
        "What is a Python list?",
        "What is a Python dictionary?",
        "What is a Python tuple?",
        "What is a Python set?",
        "What is a Python function?",
        "What is a variable in Python?",
        "What is a loop in Python?",
        "What is recursion?",
        "What is a class in Python?"
    ],

    "answer": [
        "Python is a high-level and easy-to-learn programming language.",
        "A Python list is an ordered and mutable collection of items.",
        "A Python dictionary stores data as key-value pairs.",
        "A Python tuple is an ordered collection that cannot be modified.",
        "A Python set is an unordered collection of unique values.",
        "A Python function is a reusable block of code that performs a specific task.",
        "A variable is a name used to store a value.",
        "A loop is used to repeat a block of code.",
        "Recursion is when a function calls itself to solve a problem.",
        "A class is a blueprint used to create objects."
    ]
}

dataset = Dataset.from_dict(data)

In [8]:
def format_example(example):

    messages = [
        {
            "role": "user",
            "content": example["question"]
        },
        {
            "role": "assistant",
            "content": example["answer"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}


dataset = dataset.map(format_example)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [9]:
def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [10]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

In [12]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [13]:
model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [14]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [15]:
training_args = TrainingArguments(
    output_dir="./output",

    per_device_train_batch_size=1,

    gradient_accumulation_steps=4,

    num_train_epochs=3,

    learning_rate=2e-4,

    logging_steps=1,

    save_strategy="no",

    report_to="none",

    fp16=True
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

In [17]:
trainer.train()

Step,Training Loss
1,4.097338
2,3.963423
3,3.940363
4,3.809203
5,3.481413
6,3.540633
7,3.406837
8,3.337260
9,3.031170


TrainOutput(global_step=9, training_loss=3.623070902294583, metrics={'train_runtime': 8.7439, 'train_samples_per_second': 3.431, 'train_steps_per_second': 1.029, 'total_flos': 8258429583360.0, 'train_loss': 3.623070902294583, 'epoch': 3.0})

In [18]:
save_path = "./lora_finetuned_model"

model.save_pretrained(save_path)

tokenizer.save_pretrained(save_path)

print("Model saved successfully!")

Model saved successfully!


In [19]:
!ls ./lora_finetuned_model

adapter_config.json	   chat_template.jinja	tokenizer_config.json
adapter_model.safetensors  README.md		tokenizer.json


In [20]:
def ask_model(question):

    model.eval()

    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return generated_text

In [21]:
print(
    ask_model("What is recursion?")
)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is recursion?
assistant
Recursion is a process in which a function calls itself repeatedly until it reaches a base case. The term "base case" refers to the simplest or most basic state that can be reached from within the recursive call.

Here's an example of how recursion works:

```python
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)
```

In this example, `factorial` is a recursive function that calculates the


In [22]:
print(
    ask_model("What is a Python dictionary?")
)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
What is a Python dictionary?
assistant
A Python dictionary is an unordered collection of key-value pairs. It's a data structure that allows you to store and access information in a flexible way.

Here’s how dictionaries work:

1. **Key-Value Pairs**: Each item in the dictionary is represented as a tuple where the first element is the key and the second element is the value (or sometimes, the value can be a list or another dictionary).

2. **Accessing Items**: To retrieve a value associated with a specific key,


In [ ]:
'''Cell 1  → Check GPU

Cell 2  → Install packages

Cell 3  → Import libraries

Cell 4  → Choose model

Cell 5  → Load tokenizer

Cell 6  → Load base model

Cell 7  → Create dataset

Cell 8  → Format dataset

Cell 9  → Tokenize dataset

Cell 10 → Configure LoRA

Cell 11 → Apply LoRA

Cell 12 → Data collator

Cell 13 → Training arguments

Cell 14 → Trainer

Cell 15 → Train

Cell 16 → Save model

Cell 17 → Check saved files'''
